## Introduction

Ce notebook extrait un petit sous-ensemble **stratifié** du dataset RSNA Breast Cancer Detection
et télécharge les images DICOM correspondantes depuis Kaggle.

Il réutilise directement les fonctions de `extraction_project/script/extract_download.py` — aucune duplication de code.

---

## Gestion du rate limiting (erreur 429)

Kaggle impose une limite de requêtes. Si on télécharge trop vite, il répond **HTTP 429** (Too Many Requests) avec un header `Retry-After` indiquant combien de secondes attendre.

Ce notebook utilise un algorithme **AIMD** (Additive Increase / Multiplicative Decrease) inspiré du contrôle de congestion TCP :

- **429 reçu** → pause multipliée par ×1.5 (réaction forte)
- **50 succès consécutifs** → pause réduite de −0.5s (sonde doucement la limite)
- **Gros `Retry-After` (≥ 120s)** → mode cooldown : reste conservateur 60s de plus


In [ ]:
#| echo: false
import sys, re
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / "Makefile").exists()),
    CURRENT_DIR,
)
SCRIPT_DIR = PROJECT_ROOT / "extraction_project" / "script"
SCRIPT_FILE = SCRIPT_DIR / "extract_download.py"
sys.path.insert(0, str(SCRIPT_DIR))

with open(SCRIPT_FILE, encoding="utf-8") as _f:
    _src = _f.read()

from IPython.display import Markdown, display as ipy_display

def show_func(name):
    pattern = rf'^(class {name}[\s\S]*?)(?=\n^class |\n^def |\n^# ── |\Z)'
    m = re.search(pattern, _src, re.MULTILINE)
    if not m:
        pattern = rf'^(def {name}\([\s\S]*?)(?=\n^def |\n^class |\n^# ── |\Z)'
        m = re.search(pattern, _src, re.MULTILINE)
    code = m.group(1).rstrip() if m else f"# {name} non trouvée"
    ipy_display(Markdown(f"```python\n{code}\n```"))

show_func("RateLimiter")

## Configuration


In [ ]:
import os, sys
from pathlib import Path

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [CURRENT_DIR, *CURRENT_DIR.parents] if (p / "Makefile").exists()),
    CURRENT_DIR,
)
SCRIPT_DIR = PROJECT_ROOT / "extraction_project" / "script"
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

# ── Paramètres ─────────────────────────────────────────────────────────────────
COMPETITION = "rsna-breast-cancer-detection"
BASE_DIR    = "data/extract_demo"   # dossier de destination
N_PATIENTS  = 2                     # nombre de patients (1 positif + 1 négatif minimum)
TARGET_COL  = "cancer"
GROUP_COL   = "patient_id"

os.makedirs(BASE_DIR, exist_ok=True)
print(f"Destination : {BASE_DIR}")
print(f"Patients    : {N_PATIENTS}  (garantit au moins 1 positif + 1 négatif)")

---

## Étape 1 — Authentification Kaggle

Utilise les variables d'environnement `KAGGLE_USERNAME` / `KAGGLE_KEY`
(configurées via `make setup`). Aucun token en dur.


In [ ]:
from extract_download import authenticate, ensure_csv, build_subset, prepare_tasks, download_all

api = authenticate()
print("Authentification réussie.")

## Étape 2 — Chargement du CSV


In [ ]:
import pandas as pd

csv_path = ensure_csv(api, BASE_DIR)
df = pd.read_csv(csv_path)
print(f"Dataset complet : {len(df)} images | {df[GROUP_COL].nunique()} patients")
print(f"\nDistribution '{TARGET_COL}' :")
print(df[TARGET_COL].value_counts().to_frame("count").assign(
    pct=lambda x: (x["count"] / len(df) * 100).round(2)
))

---

## Étape 3 — Sélection stratifiée

On sélectionne `N_PATIENTS` patients en conservant la proportion cancer/sain.
La fonction `build_subset` garantit qu'il y a au moins 1 positif et 1 négatif.


In [ ]:
grouped = df.groupby(GROUP_COL)[TARGET_COL].max().reset_index()
percentage = N_PATIENTS / len(grouped)

df_subset = build_subset(df, TARGET_COL, GROUP_COL, percentage)

print(f"\nPatients sélectionnés : {df_subset[GROUP_COL].nunique()}")
print(f"Images sélectionnées  : {len(df_subset)}")

In [ ]:
subset_csv = os.path.join(BASE_DIR, "train_subset.csv")
df_subset.to_csv(subset_csv, index=False)
print(f"CSV subset sauvegardé : {subset_csv}")

## Étape 4 — Téléchargement des images DICOM

Rate limiting AIMD (identique au script `extract_download.py`) :
gère automatiquement les erreurs 429 en ajustant la pause dynamiquement.


In [ ]:
tasks = prepare_tasks(df_subset, GROUP_COL, BASE_DIR)
est_gb = (len(tasks) * 5) / 1024
print(f"Images à télécharger : {len(tasks)} (~{est_gb:.3f} GB)")

In [ ]:
stats = download_all(api, tasks)

---

## Résumé


In [ ]:
print("=" * 50)
print("RÉSUMÉ")
print("=" * 50)
print(f"Dataset complet  : {df[GROUP_COL].nunique()} patients | {len(df)} images")
print(f"Subset           : {df_subset[GROUP_COL].nunique()} patients | {len(df_subset)} images")
print(f"Téléchargés      : {stats['downloaded']}")
print(f"Déjà présents    : {stats['skipped']}")
print(f"Dézippés         : {stats['unzipped']}")
print(f"Échoués          : {stats['failed']}")
print(f"CSV subset       : {subset_csv}")
print(f"Images           : {os.path.join(BASE_DIR, 'train_images')}")